In [ ]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

# ============================================
# 1. LOAD DATA & CREATE NEW COLUMN
# ============================================
df = pd.read_csv('student_performance.csv')  # adjust filename/path as needed

df['average_performance'] = (df['math score'] + df['reading score'] + df['writing score']) / 3

# ============================================
# 2. INITIALIZE APP
# ============================================
app = Dash(__name__)

# ============================================
# 3. LAYOUT
# ============================================
app.layout = html.Div([

    html.H1("Student Performance Comparison Dashboard", style={'textAlign': 'center'}),

    # ---- Filters ----
    html.Div([
        html.Div([
            html.Label("Gender"),
            dcc.Dropdown(
                id='gender-filter',
                options=[{'label': 'All', 'value': 'All'}] +
                        [{'label': g, 'value': g} for g in df['gender'].unique()],
                value='All', clearable=False
            )
        ], style={'width': '30%', 'display': 'inline-block'}),

        html.Div([
            html.Label("Lunch Type"),
            dcc.Dropdown(
                id='lunch-filter',
                options=[{'label': 'All', 'value': 'All'}] +
                        [{'label': l, 'value': l} for l in df['lunch'].unique()],
                value='All', clearable=False
            )
        ], style={'width': '30%', 'display': 'inline-block', 'marginLeft': '4%'}),

        html.Div([
            html.Label("Test Prep Course"),
            dcc.Dropdown(
                id='prep-filter',
                options=[{'label': 'All', 'value': 'All'}] +
                        [{'label': p, 'value': p} for p in df['test preparation course'].unique()],
                value='All', clearable=False
            )
        ], style={'width': '30%', 'display': 'inline-block', 'marginLeft': '4%'}),
    ], style={'padding': '20px'}),

    # ---- KPI Cards ----
    html.Div(id='kpi-cards', style={'display': 'flex', 'justifyContent': 'space-around', 'padding': '10px'}),

    # ---- Gender comparison + scatter relationship ----
    html.Div([
        dcc.Graph(id='gender-bar', style={'width': '49%', 'display': 'inline-block'}),
        dcc.Graph(id='score-scatter', style={'width': '49%', 'display': 'inline-block'}),
    ]),

    # ---- Box plot: performance by test prep ----
    dcc.Graph(id='prep-boxplot'),

])


# ============================================
# 4. CALLBACK
# ============================================
@app.callback(
    Output('kpi-cards', 'children'),
    Output('gender-bar', 'figure'),
    Output('score-scatter', 'figure'),
    Output('prep-boxplot', 'figure'),
    Input('gender-filter', 'value'),
    Input('lunch-filter', 'value'),
    Input('prep-filter', 'value'),
)
def update_dashboard(selected_gender, selected_lunch, selected_prep):

    filtered_df = df.copy()
    if selected_gender != 'All':
        filtered_df = filtered_df[filtered_df['gender'] == selected_gender]
    if selected_lunch != 'All':
        filtered_df = filtered_df[filtered_df['lunch'] == selected_lunch]
    if selected_prep != 'All':
        filtered_df = filtered_df[filtered_df['test preparation course'] == selected_prep]

    # ---- KPI Cards ----
    kpi_style = {
        'padding': '20px', 'border': '1px solid #ddd', 'borderRadius': '10px',
        'textAlign': 'center', 'width': '20%', 'boxShadow': '2px 2px 8px rgba(0,0,0,0.1)'
    }
    kpis = [
        html.Div([html.H4("Students"), html.H2(f"{len(filtered_df)}")], style=kpi_style),
        html.Div([html.H4("Avg Math"), html.H2(f"{filtered_df['math score'].mean():.1f}")], style=kpi_style),
        html.Div([html.H4("Avg Reading"), html.H2(f"{filtered_df['reading score'].mean():.1f}")], style=kpi_style),
        html.Div([html.H4("Avg Writing"), html.H2(f"{filtered_df['writing score'].mean():.1f}")], style=kpi_style),
        html.Div([html.H4("Avg Overall"), html.H2(f"{filtered_df['average_performance'].mean():.1f}")], style=kpi_style),
    ]

    # ---- Gender comparison bar chart ----
    gender_avg = filtered_df.groupby('gender')[['math score', 'reading score', 'writing score']].mean().reset_index()
    gender_melted = gender_avg.melt(id_vars='gender', var_name='subject', value_name='avg_score')
    gender_fig = px.bar(
        gender_melted, x='subject', y='avg_score', color='gender',
        barmode='group', title='Average Scores by Gender'
    )

    # ---- Scatter: Math vs Reading, colored by gender ----
    scatter_fig = px.scatter(
        filtered_df, x='math score', y='reading score', color='gender',
        title='Math vs Reading Score Relationship',
        opacity=0.6
    )

    # ---- Box plot: overall performance by test prep status ----
    box_fig = px.box(
        filtered_df, x='test preparation course', y='average_performance',
        color='test preparation course',
        title='Overall Performance Distribution by Test Preparation Status'
    )

    return kpis, gender_fig, scatter_fig, box_fig


# ============================================
# 5. RUN
# ============================================
if __name__ == '__main__':
    app.run(debug=True, port=8053)